# Lesson 5: Memory & Context Management

## What You'll Learn

1. **Why agents need memory** — The difference between stateless and stateful agents
2. **Chat history** — Remember prior questions in a session
3. **Schema cache** — Inspect once, remember for the session
4. **Context window management** — Handle token limits gracefully
5. **Session persistence** — Save and resume conversations

---

## The Memory Problem

LLMs are **stateless** — each call starts fresh with no memory of previous calls.
Everything the agent "knows" must fit in the **context window** (the input tokens).

| Memory Type | What | Where it Lives | Lifespan |
|-------------|------|----------------|----------|
| **Working memory** | Current tool results | Message history | Single turn |
| **Chat history** | Prior Q&A pairs | Injected into system prompt | Session |
| **Schema cache** | Learned data structure | In-memory dict | Session |
| **Long-term** | Cross-session knowledge | Database/file | Persistent |

### The challenge: Context windows are finite

- GPT-4o-mini: 128K tokens (~96K words)
- Claude 3.5: 200K tokens (~150K words)

Sounds huge, but a single schema inspection can be 500+ tokens, and each tool call
adds both the request and response. After 10-15 turns, you can easily hit limits.

---

## Setup

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import pandas as pd
import duckdb
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, ModelRetry

from analyst.memory.conversation import ConversationMemory, SchemaEntry
from analyst.tools.schema_inspector import inspect_schema

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
employees_df = pd.read_csv(f"{DATA_DIR}/sample_employees.csv")

print(f"Loaded: sales ({sales_df.shape}), employees ({employees_df.shape})")

Loaded: sales ((40, 8)), employees ((20, 8))


In [2]:
from pydantic_ai.exceptions import ModelHTTPError


def run_agent_checked(agent, *args, **kwargs):
    """Run agent safely and surface LM Studio template failures clearly."""
    try:
        return agent.run_sync(*args, **kwargs)
    except ModelHTTPError as exc:
        message = str(exc)
        if "No user query found in messages" in message:
            raise RuntimeError(
                "LM Studio prompt template is incompatible with this notebook flow.\n"
                "Fix in LM Studio:\n"
                "1) Use an lmstudio-community model/template for your model family.\n"
                "2) In My Models -> Prompt Template, remove logic that raises when no user query is found.\n"
                "3) Disable model thinking/reasoning output for API calls with strict structured outputs.\n"
                "4) Reload the model and rerun from the setup cell."
            ) from exc
        raise


---

## Part 1: The Problem — Stateless Agents Forget Everything

### Why can't the agent just "remember"?

Each LLM API call is **independent**. The LLM receives:
1. System prompt
2. User message
3. (Optional) message history you explicitly pass

If you don't pass prior messages, the LLM has no context. It doesn't
know what "that dataset" or "the previous result" means.

### How memory works in practice

```
Turn 1:
  User: "What columns are in the sales data?"
  Agent: "date, product, category, region, quantity, unit_price, revenue, cost"

Turn 2 (WITHOUT memory):
  User: "What is the total revenue in that dataset?"
  Agent: "I don't know which dataset you're referring to."  ← No context!

Turn 2 (WITH memory):
  System prompt now includes:
    "Prior conversation:
     User asked about sales data columns: date, product, ..."
  User: "What is the total revenue in that dataset?"
  Agent: "The total revenue in the sales dataset is $147,695.10"  ← Has context!
```

Memory = injecting prior conversation into the system prompt (or message history).

In [3]:
# A stateless agent — every call starts fresh
stateless = Agent(
    "openai:local-model",
    system_prompt="You are a data analyst. Answer concisely.",
)

# Turn 1
r1 = run_agent_checked(stateless, "The sales dataset has columns: date, product, category, region, quantity, unit_price, revenue, cost")
print(f"Turn 1: {r1.output[:100]}")

# Turn 2 — the agent has NO IDEA what "that dataset" means
r2 = run_agent_checked(stateless, "What is the total revenue in that dataset?")
print(f"Turn 2: {r2.output[:200]}")
print("\n^^^ Notice: the agent doesn't know what 'that dataset' refers to.")

Turn 1: 

Understood. I am ready to analyze your sales dataset with the columns: `date`, `product`, `categor
Turn 2: 

I cannot determine the total revenue because you haven't provided the dataset or a link to it. Please upload the file, paste the data, or specify which dataset you are referring to so I can calculat

^^^ Notice: the agent doesn't know what 'that dataset' refers to.


---

## Part 2: Adding Memory — The ConversationMemory Class

Our memory system provides:
1. **Chat history** — injected into the system prompt each turn
2. **Schema cache** — remembers data structures after first inspection
3. **Token management** — auto-trims old messages to stay within limits

### How ConversationMemory works internally

```python
class ConversationMemory:
    messages: list[Message]        # List of {role, content, timestamp}
    schema_cache: dict[str, ...]   # table_name → schema description
    max_history_tokens: int        # Token budget (default: 4000)
    max_messages: int              # Max messages to keep (default: 20)
```

**Token estimation**: Uses `len(text) // 4` as a rough approximation
(1 token ≈ 4 characters for English text). Not perfectly accurate, but
good enough for budget management. For production, use tiktoken.

**Auto-trimming**: When `add_user_message()` or `add_assistant_message()`
is called, the memory checks if total tokens exceed `max_history_tokens`.
If so, it removes the **oldest** messages first (FIFO) until under budget.

**Schema cache**: Stores table schemas separately from chat history.
Schemas are always included in the system prompt (they're small and
critical). Chat history is what gets trimmed.

### Key design decisions

| Decision | Why |
|----------|-----|
| Separate chat from schemas | Schemas are always needed; chat is expendable |
| Trim oldest first (FIFO) | Recent context is more relevant than old |
| Estimate tokens, don't count exactly | tiktoken adds a dependency; 4-char heuristic is close enough |
| Store as text, not raw messages | More portable; works across providers |

In [4]:
# -----------------------------------------------------------------
# Agent with memory — remembers prior conversation turns
# -----------------------------------------------------------------

@dataclass
class MemoryDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    data_dir: str = ""
    memory: ConversationMemory = field(default_factory=lambda: ConversationMemory(
        max_history_tokens=4000,
        max_messages=20,
    ))


class ConversationalResult(BaseModel):
    answer: str = Field(description="Answer to the current question")
    references_prior: bool = Field(description="Whether this answer builds on prior conversation")
    confidence: float = Field(ge=0.0, le=1.0)


memory_agent = Agent(
    "openai:local-model",
    deps_type=MemoryDeps,
    output_type=ConversationalResult,
    system_prompt=(
        "You are a data analyst with memory of prior conversation.\n"
        "Use the conversation history to understand follow-up questions.\n"
        "When the user says 'that', 'it', 'those', etc., refer to prior context."
    ),
    retries=2,
)


@memory_agent.system_prompt
def inject_memory(ctx: RunContext[MemoryDeps]) -> str:
    """Inject chat history and cached schemas into the system prompt."""
    parts = []

    # Add table info
    table_info = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        table_info.append(f"  '{name}': {len(df)} rows | {cols}")
    parts.append("\n".join(table_info))

    # Add cached schemas
    schemas = ctx.deps.memory.get_all_schemas_text()
    if schemas:
        parts.append(schemas)

    # Add conversation history
    history = ctx.deps.memory.get_history_text()
    if history:
        parts.append(history)

    return "\n\n".join(parts)


@memory_agent.tool
def query_data(ctx: RunContext[MemoryDeps], sql: str) -> str:
    """Run SQL against available tables."""
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(sql).fetchdf()
        if len(result) > 30:
            return f"First 30 of {len(result)} rows:\n{result.head(30).to_string()}"
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


def chat(agent, deps, question: str) -> str:
    """Send a message and update memory."""
    # Record user message
    deps.memory.add_user_message(question)

    # Run agent
    result = run_agent_checked(agent, question, deps=deps)

    # Record assistant response
    deps.memory.add_assistant_message(result.output.answer)

    return result.output


print("Memory agent ready.")

Memory agent ready.


In [5]:
# -----------------------------------------------------------------
# Multi-turn conversation — the agent remembers!
# -----------------------------------------------------------------

deps = MemoryDeps(
    tables={"sales": sales_df, "employees": employees_df},
    data_dir=DATA_DIR,
)

# Turn 1
r1 = chat(memory_agent, deps, "What is the total revenue by region?")
print(f"Turn 1: {r1.answer}\n")

# Turn 2 — follow-up referencing prior answer
r2 = chat(memory_agent, deps, "Which of those regions has the highest profit margin?")
print(f"Turn 2: {r2.answer}")
print(f"  References prior: {r2.references_prior}\n")

# Turn 3 — another follow-up
r3 = chat(memory_agent, deps, "Now break that region down by product. What sells best there?")
print(f"Turn 3: {r3.answer}")
print(f"  References prior: {r3.references_prior}")

# Check memory state
print(f"\n--- Memory Status ---")
print(f"Messages stored: {len(deps.memory.messages)}")
print(f"History tokens: ~{deps.memory.get_history_token_count()}")
print(f"Turn count: {deps.memory.session_metadata['turn_count']}")

Turn 1: The total revenue by region is as follows:

*   **East:** $39,311.20
*   **South:** $38,535.05
*   **North:** $38,010.70
*   **West:** $31,838.15

Turn 2: All regions (East, South, North, and West) have the same profit margin of 40.0%. Therefore, no single region has a higher profit margin than the others; they are all equal.
  References prior: True

Turn 3: In the East region, **Gadget Y** is the top-selling product by quantity with 490 units sold.

Here is the breakdown of sales by product in the East region, ranked by quantity:
1. **Gadget Y**: 490 units ($12,245.10 revenue)
2. **Widget A**: 360 units ($10,796.40 revenue)
3. **Gadget Z**: 215 units ($7,522.85 revenue)
4. **Gadget X**: 200 units ($2,998.00 revenue)
5. **Widget B**: 115 units ($5,748.85 revenue)

Gadget Y is the clear top seller in this region.
  References prior: True

--- Memory Status ---
Messages stored: 6
History tokens: ~231
Turn count: 3


---

## Part 3: Schema Cache — Inspect Once, Remember Forever

In [6]:
# Cache the sales schema
sales_schema = SchemaEntry(
    table_name="sales",
    row_count=len(sales_df),
    columns=[
        {"name": c, "dtype": str(sales_df[c].dtype),
         "sample_values": [str(v) for v in sales_df[c].unique()[:3]],
         "null_count": int(sales_df[c].isna().sum())}
        for c in sales_df.columns
    ],
    description="Weekly sales data with revenue and cost by product/region",
)

deps.memory.cache_schema("sales", sales_schema)

print("Schema cached. Agent will skip inspection for known tables.")
print(f"Has schema for 'sales': {deps.memory.has_schema('sales')}")
print(f"Has schema for 'employees': {deps.memory.has_schema('employees')}")
print(f"\nCached schemas text:\n{deps.memory.get_all_schemas_text()}")

Schema cached. Agent will skip inspection for known tables.
Has schema for 'sales': True
Has schema for 'employees': False

Cached schemas text:
Cached dataset schemas (already inspected):
  'sales': 40 rows | date (str), product (str), category (str), region (str), quantity (int64), unit_price (float64), revenue (float64), cost (float64)


---

## Part 4: Context Window Management

As conversations grow, we need strategies to stay within token limits.
Without management, you'll eventually hit the context window limit and
the API call will fail with a "token limit exceeded" error.

### Three strategies for managing context

```
Strategy 1: Rolling Window (simplest)
┌─────────────────────────────────┐
│ [msg1] [msg2] [msg3] ... [msgN] │  ← oldest messages dropped first
│        ────────────────→        │
│        keep last N messages     │
└─────────────────────────────────┘

Strategy 2: Summarize-and-Compact (best quality)
┌─────────────────────────────────┐
│ [summary of msgs 1-8] [msg9]   │  ← LLM summarizes old messages
│ [msg10] [msg11] [msg12]        │    into a compact paragraph
└─────────────────────────────────┘

Strategy 3: Importance-based (advanced)
┌─────────────────────────────────┐
│ [msg1:important] [msg5:numbers] │  ← keep messages with key findings
│ [msg11:recent] [msg12:recent]   │    drop chitchat
└─────────────────────────────────┘
```

### Strategy 1: Rolling window (auto-trim old messages)

In [7]:
# Simulate a long conversation
test_memory = ConversationMemory(max_history_tokens=500, max_messages=6)

for i in range(10):
    test_memory.add_user_message(f"Question {i}: What about metric {i}?")
    test_memory.add_assistant_message(f"Answer {i}: The value for metric {i} is {i * 100}.")

print(f"Added 10 Q&A pairs")
print(f"Messages retained: {len(test_memory.messages)}")
print(f"Token estimate: ~{test_memory.get_history_token_count()}")
print(f"\nRetained messages:")
for m in test_memory.messages:
    print(f"  [{m.role}] {m.content[:60]}")

Added 10 Q&A pairs
Messages retained: 6
Token estimate: ~54

Retained messages:
  [user] Question 7: What about metric 7?
  [assistant] Answer 7: The value for metric 7 is 700.
  [user] Question 8: What about metric 8?
  [assistant] Answer 8: The value for metric 8 is 800.
  [user] Question 9: What about metric 9?
  [assistant] Answer 9: The value for metric 9 is 900.


### Strategy 2: Summarize-and-compact

When the rolling window isn't enough, use an LLM to summarize the conversation
so far, then replace old messages with the summary.

In [8]:
# Use an LLM to summarize the conversation
summarizer = Agent(
    "openai:local-model",
    system_prompt=(
        "Summarize this conversation in 2-3 sentences, preserving key findings, "
        "numbers, and what the user has been asking about."
    ),
)

# Get current history and summarize
history = deps.memory.get_history_text()
if history:
    summary_result = run_agent_checked(summarizer, history)
    summary = summary_result.output

    print(f"Before compaction: {len(deps.memory.messages)} messages, ~{deps.memory.get_history_token_count()} tokens")

    # Compact: replace old messages with summary
    deps.memory.summarize_and_compact(summary)

    print(f"After compaction: {len(deps.memory.messages)} messages, ~{deps.memory.get_history_token_count()} tokens")
    print(f"\nSummary: {summary}")
else:
    print("No history to summarize yet.")

Before compaction: 6 messages, ~231 tokens
After compaction: 3 messages, ~230 tokens

Summary: 

The user initially requested total revenue by region, finding the East region generated the highest amount at $39,311.20, while all regions shared an equal 40.0% profit margin. Following this, the user asked for a product breakdown specifically within the East region to identify top sellers, revealing that Gadget Y is the best-selling product with 490 units sold.


---

## Part 5: Session Persistence — Save & Resume

In [9]:
import tempfile

# Save session
session_path = Path(tempfile.gettempdir()) / "analyst_session.json"
deps.memory.save(session_path)
print(f"Session saved to: {session_path}")

# Create a new memory and load the session
restored = ConversationMemory()
restored.load(session_path)

print(f"Restored: {len(restored.messages)} messages, {len(restored.schema_cache)} cached schemas")
print(f"Turn count: {restored.session_metadata['turn_count']}")
print(f"\nHistory preview:")
for m in restored.messages[-4:]:
    print(f"  [{m.role}] {m.content[:80]}...")

Session saved to: /var/folders/zv/hn7c_thn009b17n8mp1gghyh0000gn/T/analyst_session.json
Restored: 3 messages, 1 cached schemas
Turn count: 3

History preview:
  [assistant] [Summary of prior conversation: 

The user initially requested total revenue by ...
  [user] Now break that region down by product. What sells best there?...
  [assistant] In the East region, **Gadget Y** is the top-selling product by quantity with 490...


---

## Part 6: Using PydanticAI's Built-in Message History

PydanticAI also supports passing message history directly via `message_history`.
This is useful when you want the LLM to see the raw tool calls from prior turns.

In [10]:
# PydanticAI native message passing
simple = Agent(
    "openai:local-model",
    system_prompt="You are a data analyst. Remember prior messages.",
)

# Turn 1
r1 = run_agent_checked(simple, "The sales data has 40 rows with revenue and cost columns.")
print(f"Turn 1: {r1.output[:100]}")

# Turn 2 — pass prior messages so the agent has context
r2 = run_agent_checked(simple, 
    "Based on that data, what metrics would you compute first?",
    message_history=r1.all_messages(),  # <-- PydanticAI native history
)
print(f"Turn 2: {r2.output[:200]}")

# Turn 3 — chain further
r3 = run_agent_checked(simple, 
    "Great, now which region should we focus on?",
    message_history=r2.all_messages(),  # Includes turn 1 + 2
)
print(f"Turn 3: {r3.output[:200]}")
print(f"\nTotal messages in history: {len(r3.all_messages())}")

Turn 1: 

Understood. I have noted that the dataset contains **40 rows** with two key columns: **Revenue** a
Turn 2: 

Given a dataset with only **Revenue** and **Cost**, the most logical first step is to derive the fundamental profitability metrics. Without these, we cannot assess the financial health of the 40 tra
Turn 3: 

That is a crucial strategic question, but **we currently don't have "Region" data in the dataset you described.**

You mentioned the data has **40 rows** with only **Revenue** and **Cost**. To deter

Total messages in history: 6


### Custom memory vs PydanticAI `message_history`

| Feature | Custom `ConversationMemory` | PydanticAI `message_history` |
|---------|---------------------------|-----------------------------|
| Token management | Built-in (auto-trim) | Manual |
| Schema cache | Yes | No |
| Summarization | Yes | Manual |
| Persistence | Save/load to JSON | Manual serialization |
| Raw tool calls | No (text summaries) | Yes (full message objects) |
| Best for | Multi-turn chat UI | Agent-to-agent handoff |

**In production, combine both**: use `ConversationMemory` for the high-level context,
and `message_history` for the current turn's tool interactions.

---

## Summary

| Concept | Implementation | Why |
|---------|---------------|-----|
| Chat history | `ConversationMemory.messages` | Follow-up questions work |
| Schema cache | `ConversationMemory.schema_cache` | Avoid redundant inspections |
| Token management | Auto-trim + summarize | Stay within context limits |
| Session persistence | JSON save/load | Resume conversations |
| Native message passing | `message_history=` | Raw tool call history |

### Production patterns
1. **Always manage context size** — Unbounded history will eventually fail
2. **Cache expensive operations** — Schema inspection is the first thing to cache
3. **Summarize, don't truncate** — Summaries preserve meaning, truncation loses it
4. **Persist sessions** — Users expect to resume conversations
5. **Estimate tokens before sending** — Know how close you are to the limit

**Next: Lesson 6 — Evaluation (how to measure if your agent is actually good)**

---
## Exercises

1. **Sliding window with importance** — Keep important messages (those with numbers/findings) longer.
2. **Auto-caching** — Have the inspect tool automatically cache schemas into memory.
3. **Long-term memory** — Use SQLite to persist facts across sessions (not just within one).
4. **Context budget display** — Show the user how much context space remains.